<a href="https://colab.research.google.com/github/Titli-17/Hybrid-Movie-Recommendation-System/blob/main/Data_Prep_and_Content_Based.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
import os


def load_data(data_dir: str = "."):
    """Load movies.csv and ratings.csv from the given directory."""
    movies_path = os.path.join(data_dir, "/content/sample_data/moviedata.csv")
    ratings_path = os.path.join(data_dir, "/content/sample_data/ratings.csv")

    movies = pd.read_csv(movies_path)
    ratings = pd.read_csv(ratings_path)

    print(f"[LOAD] Movies shape : {movies.shape}")
    print(f"[LOAD] Ratings shape: {ratings.shape}")
    return movies, ratings

In [ ]:
def handle_missing_values(movies: pd.DataFrame, ratings: pd.DataFrame):
    """Step 1: Handle missing values."""
    # --- Movies ---
    print("\n--- Step 1: Handling Missing Values ---")
    print(f"  Movies  missing before:\n{movies.isnull().sum().to_string()}")
    movies["title"] = movies["title"].fillna("Unknown Title")
    movies["genres"] = movies["genres"].fillna("(no genres listed)")
    print(f"  Movies  missing after :\n{movies.isnull().sum().to_string()}")

    # --- Ratings ---
    print(f"  Ratings missing before:\n{ratings.isnull().sum().to_string()}")
    ratings = ratings.dropna(subset=["userId", "movieId", "rating"])
    print(f"  Ratings missing after :\n{ratings.isnull().sum().to_string()}")

    return movies, ratings

In [ ]:
def remove_duplicates(movies: pd.DataFrame, ratings: pd.DataFrame):
    """Step 2: Remove duplicate rows."""
    print("\n--- Step 2: Removing Duplicates ---")
    before_m, before_r = len(movies), len(ratings)
    movies = movies.drop_duplicates(subset=["movieId"])
    ratings = ratings.drop_duplicates(subset=["userId", "movieId"])
    print(f"  Movies : {before_m} -> {len(movies)}  (removed {before_m - len(movies)})")
    print(f"  Ratings: {before_r} -> {len(ratings)} (removed {before_r - len(ratings)})")
    return movies, ratings

In [ ]:
def merge_dataframes(movies: pd.DataFrame, ratings: pd.DataFrame):
    """Step 3: Merge movies and ratings into a single DataFrame."""
    print("\n--- Step 3: Merging DataFrames ---")
    merged = ratings.merge(movies, on="movieId", how="left")
    print(f"  Merged shape: {merged.shape}")
    print(f"  Columns     : {list(merged.columns)}")
    return merged

In [ ]:
def text_preprocessing(movies: pd.DataFrame):
    """
    Step 4: Text Preprocessing on the 'genres' and 'title' columns.
    - Convert genres pipe-separated list to space-separated lowercase tokens
    - Extract clean title (without year)
    - Create a combined text feature for content-based filtering
    """
    print("\n--- Step 4: Text Preprocessing ---")

    # Clean genres: "Action|Adventure|Sci-Fi" -> "action adventure scifi"
    def clean_genres(genre_str: str) -> str:
        if genre_str == "(no genres listed)":
            return ""
        genres = genre_str.lower().replace("-", "").split("|")
        return " ".join(genres)

    movies["clean_genres"] = movies["genres"].apply(clean_genres)

    # Extract year from title, e.g. "Toy Story (1995)" -> 1995
    def extract_year(title: str):
        match = re.search(r"\((\d{4})\)", title)
        return int(match.group(1)) if match else None
    movies["year"] = movies["title"].apply(extract_year)

    # Clean title: remove year and extra whitespace
    def clean_title(title: str) -> str:
        title = re.sub(r"\(\d{4}\)", "", title)  # remove (year)
        title = re.sub(r"[^a-zA-Z0-9\s]", " ", title)  # remove special chars
        return " ".join(title.lower().split())

    movies["clean_title"] = movies["title"].apply(clean_title)

    # Combined text feature for TF-IDF: clean_title + clean_genres (genres repeated for weight)
    movies["text_features"] = movies["clean_title"] + " " + movies["clean_genres"] + " " + movies["clean_genres"]

    print(f"  Sample text_features:\n{movies[['title', 'text_features']].head(3).to_string()}")
    return movies

In [ ]:
def preprocess(data_dir: str = "."):
    """
    Full preprocessing pipeline.
    Returns:
        movies  : cleaned movies DataFrame (with text features)
        ratings : cleaned ratings DataFrame
        merged  : merged DataFrame
    """
    print("=" * 60)
    print("  DATA PREPROCESSING & CLEANING")
    print("=" * 60)

    movies, ratings = load_data(data_dir)
    movies, ratings = handle_missing_values(movies, ratings)
    movies, ratings = remove_duplicates(movies, ratings)
    merged = merge_dataframes(movies, ratings)
    movies = text_preprocessing(movies)

    print("\n[DONE] Preprocessing complete.\n")
    return movies, ratings, merged


if __name__ == "__main__":
    movies, ratings, merged = preprocess(".")
    print(movies.head())
    print(ratings.head())

In [ ]:
# Execute the full pipeline
movies_raw, ratings_raw = load_data()

# 1. Handle missing values
movies, ratings = handle_missing_values(movies_raw.copy(), ratings_raw.copy())

# 2. Remove duplicates
movies, ratings = remove_duplicates(movies, ratings)

# 3. Merge dataframes
merged_df = merge_dataframes(movies, ratings)

# 4. Text Preprocessing
movies = text_preprocessing(movies)

print("\n--- Pipeline Complete ---")
print(f"Final Movies DataFrame shape: {movies.shape}")

In [ ]:
"""
Content-Based Filtering Module
================================
Flowchart path:
  Content-Based Model (Movie Features & Description)
    -> TF-IDF Vectorization (Convert Text to Data)
    -> Cosine Similarity (Movie Similarity)
"""

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class ContentBasedRecommender:
    def __init__(self):
        self.tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2))
        self.tfidf_matrix = None
        self.movies = None
        self.movie_indices = None

    def fit(self, movies: pd.DataFrame):
        self.movies = movies.reset_index(drop=True)
        self.tfidf_matrix = self.tfidf_vectorizer.fit_transform(self.movies['text_features'])
        self.movie_indices = pd.Series(self.movies.index, index=self.movies['title'])

    def get_similar_movies(self, title: str, top_n: int = 10) -> pd.DataFrame:
        if title not in self.movie_indices.index:
            matches = self.movies[self.movies['title'].str.contains(title, case=False, na=False)]
            if matches.empty: return pd.DataFrame()
            title = matches.iloc[0]['title']
        idx = self.movie_indices[title]
        if isinstance(idx, pd.Series): idx = idx.iloc[0]
        sim_scores = cosine_similarity(self.tfidf_matrix[idx : idx + 1], self.tfidf_matrix).flatten()
        sim_indices = np.argsort(sim_scores)[::-1][1 : top_n + 1]
        results = self.movies.iloc[sim_indices][['movieId', 'title', 'genres']].copy()
        results['cb_score'] = sim_scores[sim_indices]
        return results.reset_index(drop=True)

    def recommend_for_user(self, user_id: int, ratings: pd.DataFrame, movies: pd.DataFrame, top_n: int = 10) -> pd.DataFrame:
        user_ratings = ratings[ratings['userId'] == user_id]
        liked = user_ratings[user_ratings['rating'] >= 4.0].sort_values('rating', ascending=False)
        if liked.empty: liked = user_ratings.sort_values('rating', ascending=False).head(5)
        already_seen = set(user_ratings['movieId'].values)
        candidate_scores = {}
        for _, row in liked.head(5).iterrows():
            movie_match = self.movies[self.movies['movieId'] == row['movieId']]
            if movie_match.empty: continue
            similar = self.get_similar_movies(movie_match.iloc[0]['title'], top_n=20)
            for _, s_row in similar.iterrows():
                mid = s_row['movieId']
                if mid not in already_seen:
                    candidate_scores[mid] = candidate_scores.get(mid, 0) + (row['rating'] * s_row['cb_score'])
        if not candidate_scores: return pd.DataFrame()
        result_ids = sorted(candidate_scores, key=candidate_scores.get, reverse=True)[:top_n]
        results = self.movies[self.movies['movieId'].isin(result_ids)][['movieId', 'title', 'genres']].copy()
        results['cb_score'] = results['movieId'].map(candidate_scores)
        return results.sort_values('cb_score', ascending=False).reset_index(drop=True)

# Execution
cb = ContentBasedRecommender()
cb.fit(movies)
print('\n--- Similar to Toy Story ---')
print(cb.get_similar_movies('Toy Story (1995)', top_n=5))
print('\n--- Recommendations for User 1 ---')
print(cb.recommend_for_user(1, ratings, movies, top_n=5))

In [ ]:
import matplotlib.pyplot as plt

# Example data (use your actual output)
movies = ["Toy Story 2", "Moana", "Antz", "Toy Story 3", "The Good Dinosaur"]
scores = [1.0, 0.8356, 0.8356, 0.8222, 0.7578]

plt.figure(figsize=(8,5))
plt.bar(movies, scores)

plt.xlabel("Movies")
plt.ylabel("Similarity Score")
plt.title("Content-Based Similarity Scores")

plt.xticks(rotation=45)
plt.grid()

plt.savefig("content_similarity.png")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.barh(movies, scores)

plt.xlabel("Similarity Score")
plt.ylabel("Movies")
plt.title("Top Recommended Movies (Content-Based)")

plt.grid()
plt.savefig("top_recommendations.png")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(scores, bins=5)

plt.xlabel("Similarity Score")
plt.ylabel("Frequency")
plt.title("Distribution of Similarity Scores")

plt.grid()
plt.savefig("similarity_distribution.png")
plt.show()